# UniChecksum — Python quickstart

`unichecksum` is a Cython extension over the UniChecksum C ABI, shipped as a
self-contained wheel: the native library travels inside the package, so
installing it needs neither Nim nor a compiler. The distribution is
`lituus-unichecksum`; the import name is not namespaced.

```
pip install lituus-unichecksum
```

CI executes this notebook against the wheel the release actually publishes, so
an output below that stops matching fails the build.

## The API

In [1]:
import unichecksum

unichecksum.version(), unichecksum.__version__

('0.2.0', '0.2.0')

Three families, each pinned to the parameters its consuming format
mandates. `"123456789"` is the input every CRC catalogue publishes a check value
for, which makes it the first thing to try against any implementation.

In [2]:
data = b"123456789"
{
    "crc32": hex(unichecksum.crc32(data)),
    "crc64": hex(unichecksum.crc64(data)),
    "adler32": hex(unichecksum.adler32(data)),
}

{'crc32': '0xcbf43926', 'crc64': '0x995dc9bbdf1939fa', 'adler32': '0x91e01de'}

## zlib agrees

CRC-32 and Adler-32 are the two the standard library also implements, which
makes `zlib` a free independent check.

In [3]:
import zlib

(unichecksum.crc32(data) == zlib.crc32(data),
 unichecksum.adler32(data) == zlib.adler32(data))

(True, True)

## Continuing a previous result

The second argument is a checksum to continue from, the same convention as
`zlib.crc32`, so data arriving in pieces needs no separate state object.

In [4]:
first, second = b"12345", b"6789"
running = unichecksum.crc32(first)
unichecksum.crc32(second, running) == unichecksum.crc32(first + second)

True

## The empty input is not zero everywhere

A CRC starts at all-ones and ends by inverting, so with nothing in between the
two cancel. Adler-32 starts its low half at one and never transforms its output,
so the empty input keeps that one — treating zero as "no checksum computed" is
safe for the CRCs and wrong for Adler-32.

In [5]:
(unichecksum.crc32(b""), unichecksum.crc64(b""), unichecksum.adler32(b""))

(0, 0, 1)

## Any bytes-like object

A contiguous buffer is read in place, without being copied. A strided
memoryview cannot be, so it is copied once instead — the result is the same
either way.

In [6]:
import array

strided = memoryview(b"abcdef")[::2]
(unichecksum.crc32(bytearray(data)) == unichecksum.crc32(data),
 unichecksum.crc32(array.array("B", data)) == unichecksum.crc32(data),
 unichecksum.crc32(strided) == unichecksum.crc32(b"ace"))

(True, True, True)

A `str` has no bytes until it is encoded, so it is refused rather
than guessed at.

In [7]:
try:
    unichecksum.crc32("123456789")
except TypeError as exc:
    print("TypeError:", exc)

TypeError: data must be bytes-like, not str


## The C ABI underneath

The same entry points are reachable from anything that speaks C. There the
contract is expressed by clamping instead of raising — an exception must never
unwind across an ABI boundary:

```c
unichecksum_crc32(NULL, 8);                 /* 0     — nothing was read */
unichecksum_crc32_update(state, NULL, 8);   /* state — unchanged */
```

See `include/UniChecksum.h`, and the book for the full picture.